# Machine Learning Assignment

**Name:** Shivanii M  
**BITS ID:** 2026CT05079  
**Course:** M.Tech - AIML

---

1. Importing Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split

2. Loading the Dataset

In [ ]:
csv_path = Path("data/Myocardial infarction complications Database.csv")
if not csv_path.exists():
    csv_path = Path("/content/data/Myocardial infarction complications Database.csv")

df_raw = pd.read_csv(csv_path)
print("Shape:", df_raw.shape)
df_raw.head()

3. Target Variable & Data Exploration (EDA)

In [ ]:
# target variable: 0 = alive, 1 = lethal outcome (any cause 1-7)
print("Original LET_IS counts:\n", df_raw["LET_IS"].value_counts().sort_index())
y = (df_raw["LET_IS"] > 0).astype(int)

print("\nBinary target distribution:")
print(y.value_counts())
print("Proportions:\n", y.value_counts(normalize=True).round(4))
print("Imbalance ratio:", round((y == 0).sum() / (y == 1).sum(), 2))

# basic EDA checks required by assignment
print("\nDuplicate rows:", df_raw.duplicated().sum())
print("Total missing cells:", df_raw.isna().sum().sum())
print("Data types:\n", df_raw.dtypes.value_counts())

3.1 Class Distribution Plot

In [ ]:
class_counts = y.value_counts().sort_index()

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Survived (0)", "Died (1)"], class_counts.values, color=["#4c78a8", "#e45756"])
for i, v in enumerate(class_counts.values):
    ax.text(i, v + 15, str(v), ha="center")
ax.set_ylabel("Number of patients")
ax.set_title("Class distribution of the target")
plt.tight_layout()
plt.show()

4. Feature Selection & Missing Value Audit

In [ ]:
id_col = "ID"
other_outcomes = list(df_raw.columns[112:123])  # remaining 11 complication columns
leak_48h = ["R_AB_3_n", "NA_R_3_n", "NOT_NA_3_n"]  # day-3 features excluded for 48h prediction
assert list(df_raw.columns[[94, 101, 104]]) == leak_48h

X = df_raw.drop(columns=[id_col, "LET_IS"] + other_outcomes + leak_48h)
print("Features before missing filter:", X.shape)

# audit missing value percentages
miss_pct = (X.isna().mean() * 100).sort_values(ascending=False)
print("\nTop 12 columns by missing %:\n", miss_pct.head(12).round(2))
print("Overall missing %:", round(X.isna().sum().sum() / X.size * 100, 2))

top_missing = miss_pct.head(15)
plt.figure(figsize=(7, 5))
sns.barplot(x=top_missing.values, y=top_missing.index, color="#4c78a8")
plt.axvline(50, color="red", linestyle="--", label="50% cutoff")
plt.xlabel("% missing")
plt.ylabel("Feature")
plt.title("Columns with most missing values")
plt.legend()
plt.tight_layout()
plt.show()

# drop columns with >50% missing values
too_empty = miss_pct[miss_pct > 50].index.tolist()
print("Dropping (>50% missing):", too_empty)
X = X.drop(columns=too_empty)

# check and drop exact duplicate rows if any exist
dup_mask = X.duplicated()
print("Duplicate rows:", dup_mask.sum())
if dup_mask.sum() > 0:
    X = X[~dup_mask]
    y = y[~dup_mask]

print(f"Final clean shape -> Instances: {X.shape[0]} (>=1000: {X.shape[0] >= 1000}) | Features: {X.shape[1]} (>=12: {X.shape[1] >= 12})")

5. Train-Test Split (80:20 Stratified) & Exporting

In [ ]:
# 80:20 split keeping the same class ratio (84% vs 16%) and a fixed seed
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print("Train shape:", X_train.shape, "| Positive rate:", round(y_train.mean(), 4))
print("Test shape :", X_test.shape,  "| Positive rate:", round(y_test.mean(), 4))

# save the 20% test set for the GitHub repository requirement
test_df = X_test.copy()
test_df["target_LET_IS"] = y_test.values
test_df.to_csv("test_data.csv", index=False)
print("Saved test_data.csv ->", test_df.shape)

6. Feature Grouping (Binary, Continuous, Ordinal, Categorical)

In [ ]:
# inspect unique levels on X_train only to avoid data leakage
levels = X_train.nunique()

binary_cols = levels[levels <= 2].index.tolist()
continuous_cols = levels[levels > 10].index.tolist()
coded_cols = levels[(levels > 2) & (levels <= 10)].index.tolist()

# ZSN_A is partially ordered and the 4 ECG columns are QRS pattern categories -> one-hot encode
onehot_cols = ["ZSN_A", "ant_im", "lat_im", "inf_im", "post_im"]
ordinal_cols = [c for c in coded_cols if c not in onehot_cols]

type_summary = pd.DataFrame({
    "feature type": ["binary (0/1)", "continuous", "ordinal codes", "categorical codes (one-hot)"],
    "count": [len(binary_cols), len(continuous_cols), len(ordinal_cols), len(onehot_cols)]
})
print(type_summary)
print("Total:", type_summary["count"].sum(), "| X_train cols:", X_train.shape[1])
print("Continuous:", continuous_cols)
print("Ordinal:", ordinal_cols)

7. Leak-Free Preprocessing Pipeline (Imputation, Encoding & Scaling)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# continuous columns: median imputation + standard scaling
continuous_pipe = Pipeline([
    ("fill", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

# ordinal columns: mode imputation + standard scaling
ordinal_pipe = Pipeline([
    ("fill", SimpleImputer(strategy="most_frequent")),
    ("scale", StandardScaler()),
])

# binary columns: mode imputation only
binary_pipe = Pipeline([
    ("fill", SimpleImputer(strategy="most_frequent")),
])

# categorical columns: mode imputation + one-hot encoding
onehot_pipe = Pipeline([
    ("fill", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("cont", continuous_pipe, continuous_cols),
    ("ordi", ordinal_pipe, ordinal_cols),
    ("bina", binary_pipe, binary_cols),
    ("cate", onehot_pipe, onehot_cols),
], verbose_feature_names_out=False)
preprocessor.set_output(transform="pandas")

# fit only on training data, then transform both splits
X_train_p = preprocessor.fit_transform(X_train)
X_test_p = preprocessor.transform(X_test)

print("After preprocessing:", X_train_p.shape, X_test_p.shape)
print("Missing before (train/test):", X_train.isna().sum().sum(), X_test.isna().sum().sum())
print("Missing after  (train/test):", X_train_p.isna().sum().sum(), X_test_p.isna().sum().sum())
print("Train max abs mean:", X_train_p[continuous_cols].mean().round(3).abs().max())
print("Test means:", X_test_p[continuous_cols].mean().round(3).to_dict())

8. Training All 5 Classification Models & Saving to model/

In [ ]:
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

my_seed = 42

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=my_seed),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, min_samples_leaf=10, class_weight="balanced", random_state=my_seed),
    "kNN": KNeighborsClassifier(n_neighbors=7),
    "Naive Bayes": GaussianNB(),
    "Random Forest (Ensemble)": RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=5,
        class_weight="balanced", random_state=my_seed, n_jobs=-1
    ),
}

# create model/ directory and save each fitted model
model_dir = Path("model") / "saved_models"
model_dir.mkdir(parents=True, exist_ok=True)
plot_dir = Path("model") / "plots"
plot_dir.mkdir(parents=True, exist_ok=True)

for name, model in models.items():
    model.fit(X_train_p, y_train)
    clean_name = name.lower().replace(" ", "_").replace("(", "").replace(")", "")
    joblib.dump(model, model_dir / f"{clean_name}.joblib")

print("Saved models:", sorted([f.name for f in model_dir.glob("*.joblib")]))

9. Evaluating All 5 Classification Models (6 Required Metrics)

In [ ]:
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef
)

rows = []
test_pred = {}
test_proba = {}

for name, model in models.items():
    pred = model.predict(X_test_p)
    proba = model.predict_proba(X_test_p)[:, 1]
    test_pred[name] = pred
    test_proba[name] = proba

    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "AUC": roc_auc_score(y_test, proba),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "MCC": matthews_corrcoef(y_test, pred),
    })

results = pd.DataFrame(rows).set_index("Model").round(4)
print(results)
print("\nPredicted deaths:", {k: int(v.sum()) for k, v in test_pred.items()}, "| Actual:", int(y_test.sum()))

10. Visualization 1 - Grouped Bar Chart of All 6 Metrics

In [ ]:
ax = results.plot(kind="bar", figsize=(12, 6), width=0.8, colormap="tab10")

ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylim(-0.1, 1.1)
ax.set_ylabel("Score")
ax.set_xlabel("")
ax.set_title("Test Set Performance Metrics Across All Five Models")
ax.legend(title="Metric", loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=6)
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=15)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", fontsize=7, padding=2)

plt.tight_layout()
plt.savefig(plot_dir / "metrics_comparison.png", dpi=150)
plt.show()

11. Visualization 2 - ROC Curves Comparison

In [ ]:
from sklearn.metrics import roc_curve

plt.figure(figsize=(7, 6))

for name, proba in test_proba.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    plt.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC = {results.loc[name, 'AUC']:.3f})")

plt.plot([0, 1], [0, 1], "k--", label="Random guess")

plt.xlabel("False positive rate")
plt.ylabel("True positive rate (recall)")
plt.title("ROC Curves of All Five Models")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(plot_dir / "roc_curves.png", dpi=150)
plt.show()

12. Visualization 3 - Confusion Matrices Side-by-Side

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for ax, (name, pred) in zip(axes, test_pred.items()):
    cm = confusion_matrix(y_test, pred)
    ConfusionMatrixDisplay(cm, display_labels=["Survived", "Died"]).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name, fontsize=10)

plt.tight_layout()
plt.savefig(plot_dir / "confusion_matrices.png", dpi=150)
plt.show()

13. Model Ranking Summary, Saving Preprocessor & Environment Versions

In [ ]:
import sklearn
import matplotlib as mpl

# save fitted preprocessor with the other saved estimators
joblib.dump(preprocessor, model_dir / "preprocessor.joblib")
results.to_csv("results_table.csv")

print("Saved model files:", sorted([f.name for f in model_dir.iterdir()]))

print("\nBest model per metric:")
for metric in results.columns:
    print(f"  {metric:10s} -> {results[metric].idxmax()} ({results[metric].max():.4f})")

print("\nOverall average score ranking:")
print(results.mean(axis=1).round(4).sort_values(ascending=False))

print("\nEnvironment versions for requirements.txt:")
print(f"pandas=={pd.__version__}\nnumpy=={np.__version__}\nscikit-learn=={sklearn.__version__}\n"
      f"matplotlib=={mpl.__version__}\nseaborn=={sns.__version__}\njoblib=={joblib.__version__}")

In [ ]:
import shutil
from google.colab import files

# To create a clean staging folder with everything generated in Colab
!mkdir -p export_bundle/model
!cp test_data.csv results_table.csv model/plots/*.png export_bundle/
!cp model/saved_models/*.joblib export_bundle/model/

# zipping everything to download all together.
shutil.make_archive("assignment_files", "zip", "export_bundle")
files.download("assignment_files.zip")